In [ ]:
from datasets import load_from_disk
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance
from tqdm import tqdm
import uuid
from dotenv import load_dotenv
import os

load_dotenv()

In [7]:
dataset = load_from_disk("hotpot_mini_1k")

print(len(dataset))
print(dataset[0].keys())

1000
dict_keys(['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context'])


In [9]:
print(dataset[0]["context"])

{'sentences': [['Jesse Oren Kellerman (born September 1, 1978) is an American novelist and playwright.', ' He has published the novels: "Sunstroke" (2006), "Trouble" (2007), "The Genius" (2008), "The Executor" (2010), "I\'ll Catch You" (2012), and with his father, Jonathan Kellerman, "The Golem of Hollywood" (2014).', ' For his play, "Things Beyond Our Control" (2004) he was honored with a Princess Grace Award, which recognizes emerging talent in theater, dance, and film in the U.S.'], ['Ah, But Your Land is Beautiful is the third novel of Alan Paton, the South African author who is best known for writing "Cry, the Beloved Country".', ' "Ah, but Your Land is Beautiful" is an anti-apartheid novel, in a similar vein to "Cry, the Beloved Country".', " It is a fictional reworking of Paton's own years working as a political activist and of the experience he gained working as the president of the South African Liberal Party."], ['Andrew Brown is a South African novelist influenced by William

In [10]:
documents = []

for item in dataset:

    question = item["question"]

    titles = item["context"]["title"]
    sentences_list = item["context"]["sentences"]

    for title, sentences in zip(titles, sentences_list):

        text = " ".join(sentences)

        documents.append({
            "text": text,
            "title": title,
            "question": question
        })

print("Total docs:", len(documents))

Total docs: 9948


In [17]:
model = SentenceTransformer("all-MiniLM-L6-v2")

In [26]:
import os
from openai import OpenAI

deepseek = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

In [18]:
texts = [doc["text"] for doc in documents]

embeddings = model.encode(
    texts,
    show_progress_bar=True,
    batch_size=64
)

print("Embedding shape:", len(embeddings))

Batches:   0%|          | 0/156 [00:00<?, ?it/s]

Embedding shape: 9948


In [19]:
collection_name = "hotpot_docs"

client.recreate_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(
        size=384,
        distance=Distance.COSINE
    )
)

print("Collection created")

Collection created


C:\Users\ASUS\AppData\Local\Temp\ipykernel_23972\3825619832.py:3: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


In [20]:
points = []

for i, (doc, emb) in enumerate(zip(documents, embeddings)):

    points.append({
        "id": i,
        "vector": emb.tolist(),
        "payload": {
            "text": doc["text"],
            "title": doc["title"],
            "question": doc["question"]
        }
    })

print("Points created:", len(points))

Points created: 9948


In [22]:
from tqdm import tqdm

batch_size = 256

for i in tqdm(range(0, len(points), batch_size)):
    
    batch = points[i:i+batch_size]

    client.upsert(
        collection_name=collection_name,
        points=batch
    )

print("Inserted:", len(points))

100%|██████████| 39/39 [00:02<00:00, 17.83it/s]

Inserted: 9948


In [24]:
query = "Who wrote Cry the Beloved Country?"

query_vector = model.encode(query).tolist()

results = client.query_points(
    collection_name=collection_name,
    query=query_vector,
    limit=3
)

for r in results.points:
    print("\nScore:", r.score)
    print(r.payload["text"])


Score: 0.6618629
Ah, But Your Land is Beautiful is the third novel of Alan Paton, the South African author who is best known for writing "Cry, the Beloved Country".  "Ah, but Your Land is Beautiful" is an anti-apartheid novel, in a similar vein to "Cry, the Beloved Country".  It is a fictional reworking of Paton's own years working as a political activist and of the experience he gained working as the president of the South African Liberal Party.

Score: 0.6365688
Alan Stewart Paton (11 January 1903 – 12 April 1988) was a South African author and anti-apartheid activist.  His works include the novels "Cry, the Beloved Country" and "Too Late the Phalarope".

Score: 0.52128506
"Sad Movies (Make Me Cry)" is a 1961 pop song by the American singer Sue Thompson.  The song was written by John D. Loudermilk and appears on Thompson's 1962 Hickory Records album "Meet Sue Thompson".
